# Reasoning models: thinking off vs on

This notebook goes with the [Reasoning model](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/reasoning-model.html) dictionary entry. It asks Claude one question with a checkable answer, at several thinking settings. For each run it shows:

1. **What you're billed for:** `output_tokens`, and how much of that was thinking (`usage.output_tokens_details.thinking_tokens`).
2. **What you can read:** the answer, and for the `high` run, a thinking *summary*.
3. **How much it varies:** run it more than once and the counts change, even with the same prompt and settings.

**You need your own Anthropic API key**, and the calls are billed to your account. With the defaults (Claude Haiku 5.5, 2 repeats, 10 calls), it costs well under a cent. On Opus 5.5, expect about 40 times that, which is still a few cents.

## 1. Install the SDK

Colab already has `pandas` and `matplotlib`. This adds the Anthropic Python SDK.

In [ ]:
%pip install -q anthropic

## 2. Add your API key (without saving it in the notebook)

**Recommended: Colab Secrets.** Click the 🔑 icon in the left sidebar, add a secret named `ANTHROPIC_API_KEY`, paste your key as its value, and turn on **Notebook access**. The key stays in your Colab account, not in this notebook.

**Otherwise:** the cell below asks for the key in a hidden prompt. It lasts only for this session.

Don't paste your key into a code cell. Saved notebooks keep the contents of their cells.

In [ ]:
import os, getpass
import anthropic

try:                                    # Colab Secrets, if you've set one up
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

client = anthropic.Anthropic()
print("Key loaded (not shown). anthropic SDK", anthropic.__version__)

## 3. Choose your settings

Use the form on the right, or edit the values. The default question is a Simpson's paradox: store B converts better on mobile *and* on desktop, but store A converts better overall, because the two stores get different mixes of traffic. **Correct answer: mobile: B, desktop: B, overall: A.** If you write your own question, give it an answer you can check.

Prices are standard per-million-token rates from the [Anthropic pricing page](https://platform.claude.com/docs/en/about-claude/pricing), read 2026-11-25. Check the page before you rely on them, because prices change.

In [ ]:
MODEL = "claude-haiku-5-5"  # @param ["claude-haiku-5-5", "claude-sonnet-5-5", "claude-opus-5-5"]
QUESTION = "Store A: mobile 10 conversions from 100 visits, desktop 300 from 400. Store B: mobile 60 conversions from 400 visits, desktop 80 from 100. Which store has the higher conversion rate on mobile, on desktop, and overall? End with one line: 'mobile: X, desktop: X, overall: X'."  # @param {type:"string"}
TRIVIAL = "What is the capital of France? One word."  # @param {type:"string"}
REPEATS = 2  # @param {type:"slider", min:1, max:5, step:1}
MAX_TOKENS = 8000  # @param {type:"integer"}

# $ per million tokens: (input, output). Haiku 5.5 rates are for prompts up to 100K tokens.
PRICES = {"claude-haiku-5-5": (0.10, 0.50),
          "claude-sonnet-5-5": (2.00, 10.00),
          "claude-opus-5-5": (4.00, 20.00)}
PRICE_IN, PRICE_OUT = PRICES[MODEL]

## 4. The runs

This is the same code as [`thinking_api.py`](https://github.com/bradleyboehmke/ai-engineering-notes/blob/main/learning/llm-mechanics/experiments/thinking_api.py). The only change is that the "off" run depends on the model, because each one turns thinking off differently, or can't turn it off at all:

| Model | Thinking off | Source |
|---|---|---|
| Haiku 5.5 | `thinking: {"type": "disabled"}`, at `high` effort or below | [Effort](https://platform.claude.com/docs/en/build-with-claude/effort) |
| Sonnet 5.5 | `"disabled"` returns a 400 error. `thinking: {"type": "between_tools"}` turns off up-front thinking instead. | [Effort](https://platform.claude.com/docs/en/build-with-claude/effort) |
| Opus 5.5 | Not possible. Every effort level returns a 400 error, so the notebook skips this run. | [Effort](https://platform.claude.com/docs/en/build-with-claude/effort) |

The other runs only set `effort`, except `high`, which also asks for a thinking summary (`display: "summarized"`). On current models the default display is `"omitted"`: you get an empty thinking block and pay for the full thinking anyway.

In [ ]:
import time

OFF = {"claude-haiku-5-5": {"thinking": {"type": "disabled"}},
       "claude-sonnet-5-5": {"thinking": {"type": "between_tools"}},
       "claude-opus-5-5": None}                       # can't turn thinking off

RUNS = [("1. off",     QUESTION, OFF[MODEL]),
        ("2. low",     QUESTION, {"output_config": {"effort": "low"}}),
        ("3. medium",  QUESTION, {"output_config": {"effort": "medium"}}),
        ("4. high",    QUESTION, {"output_config": {"effort": "high"},
                                  "thinking": {"type": "adaptive", "display": "summarized"}}),
        ("5. trivial", TRIVIAL,  {})]                  # model's default effort


def run_once(label, prompt, extra):
    t0 = time.perf_counter()
    r = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,            # thinking counts toward this cap too
        messages=[{"role": "user", "content": prompt}],
        **extra,
    )
    secs = time.perf_counter() - t0
    u = r.usage
    think = u.output_tokens_details.thinking_tokens if u.output_tokens_details else 0
    text = next((b.text for b in r.content if b.type == "text"), "")
    summary = next((b.thinking for b in r.content if b.type == "thinking"), "")
    return {
        "run": label,
        "in": u.input_tokens,
        "out": u.output_tokens,           # what you're billed for
        "think": think,                   # the part of `out` that was thinking
        "text": u.output_tokens - think,  # the part you can read
        "blocks": "+".join(b.type for b in r.content),
        "sec": round(secs, 1),
        "cost_$": (u.input_tokens * PRICE_IN + u.output_tokens * PRICE_OUT) / 1e6,
        "stop": r.stop_reason,
        "last_line": text.strip().splitlines()[-1] if text.strip() else "(no text)",
        "summary": summary,
    }

## 5. Run them

This cell makes `5 × REPEATS` API calls, or one fewer per repeat on Opus 5.5.

In [ ]:
import pandas as pd

rows = []
for rep in range(1, REPEATS + 1):
    for label, prompt, extra in RUNS:
        if extra is None:
            print(f"{label}: skipped. {MODEL} can't turn thinking off.")
            continue
        row = run_once(label, prompt, extra)
        row["repeat"] = rep
        rows.append(row)

df = pd.DataFrame(rows)
cols = ["repeat", "run", "in", "out", "think", "text", "blocks", "sec", "cost_$", "stop", "last_line"]
pd.set_option("display.max_colwidth", 60)
df[cols].style.format({"cost_$": "{:.6f}"}).hide(axis="index")

## 6. Billed vs visible

Each bar is one call. The full bar is `output_tokens`, which is what you pay for. The hatched part is thinking you didn't get to read.

In [ ]:
import matplotlib.pyplot as plt

labels = [f"{r} · #{k}" for r, k in zip(df["run"], df["repeat"])]
fig, ax = plt.subplots(figsize=(8, 0.42 * len(df) + 1))
ax.barh(labels, df["text"], color="#8a4620", label="visible text")
ax.barh(labels, df["think"], left=df["text"], color="white", edgecolor="#8a4620",
        hatch="///", label="thinking (billed, not shown)")
ax.invert_yaxis()
ax.set_xlabel("output tokens (billed)")
ax.set_title(f"{MODEL}: what you pay for vs what you read")
ax.legend(loc="lower right", frameon=False)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

totals = df.groupby("run")[["out", "think", "cost_$"]].agg(["min", "max"])
print("Spread across repeats (min, max):")
print(totals.to_string())

## 7. The thinking summary

Only the `high` run asked for a summary. Compare its length with that run's `think` count. You're billed for the raw thinking, not for this summary, and the summary is written by a different model.

In [ ]:
high = df[df["run"] == "4. high"].iloc[-1]
print(f"Billed thinking tokens: {high['think']}\n")
print(high["summary"] or "(no thinking block)")

## What to look for

- **Correctness:** did every run end with `mobile: B, desktop: B, overall: A`? If the "off" run was right too, then thinking changed the bill on this question but not the answer.
- **The bill:** what share of `out` was `think` at each effort level? Does `high` always cost more than `low`, or does that change between repeats?
- **Run-to-run spread:** compare the min and max for the same setting. Is the gap between repeats smaller or larger than the gap between effort levels?
- **The trivial question:** did the model skip thinking (`think` = 0)?
- **Try a harder question** of your own, or change `MODEL`, and see which of these hold.

Back to the [Reasoning model](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/reasoning-model.html) entry · the same code as a script: [`thinking_api.py`](https://github.com/bradleyboehmke/ai-engineering-notes/blob/main/learning/llm-mechanics/experiments/thinking_api.py)